# Основы Cython — от Python до C

**Цель:** понять, как Cython превращает Python-код в C,
что даёт статическая типизация и где находятся границы применимости.

---
## Содержание
1. Что такое Cython и как он работает
2. Первая `%%cython` ячейка
3. `cdef` — статические типы
4. `def` / `cdef` / `cpdef`
5. Typed memoryviews
6. `nogil` и директивы компилятора
7. Бенчмарки (работают без C-компилятора)
8. Ограничения Cython

---
## 0  Установка

```bash
pip install cython numpy matplotlib
# C-компилятор для %%cython ячеек:
# Windows: https://visualstudio.microsoft.com/visual-cpp-build-tools/
# Linux:   sudo apt install gcc
# macOS:   xcode-select --install
```

In [ ]:
# Проверяем наличие Cython и C-компилятора
import sys, subprocess
try:
    import Cython
    print(f'Cython {Cython.__version__} OK')
except ImportError:
    print('Cython не установлен: pip install cython')

r = subprocess.run(['gcc', '--version'], capture_output=True, text=True)
if r.returncode == 0:
    print('gcc OK:', r.stdout.split('\n')[0])
else:
    r2 = subprocess.run(['cl'], capture_output=True, text=True)
    if r2.returncode == 0 or 'Microsoft' in r2.stderr:
        print('MSVC OK')
    else:
        print('C-компилятор не найден — %%cython ячейки не запустятся')
        print('Бенчмарки в разделе 7 работают без компилятора')

---
## 1  Что такое Cython?

```
  .pyx / %%cython  -->  [cython]  -->  .c  -->  [gcc/MSVC]  -->  .so / .pyd
                                                                      |
                                                            import как обычный модуль
```

Cython — **надмножество Python**. Любой валидный Python — валидный Cython.

| Синтаксис | Что делает |
|-----------|------------|
| `cdef int x` | переменная как C `int`, не PyObject |
| `cdef double[:] a` | typed memoryview — указатель на буфер NumPy |
| `cdef func()` | C-функция, не видна из Python |
| `cpdef func()` | C-функция + Python-обёртка |
| `with nogil:` | блок без GIL (только C-типы внутри) |

### Почему Python медленный в числодробилках?

Каждый Python `float` — это **heap-объект** размером 24 байта:

```c
typedef struct {
    Py_ssize_t   ob_refcnt;   // 8 байт — счётчик ссылок
    PyTypeObject *ob_type;    // 8 байт — указатель на тип
    double        ob_fval;    // 8 байт — само значение
} PyFloatObject;
```

Операция `a + b` в Python — 6 шагов вместо 1:

```
1. PyNumber_Add(a, b)      <- vtable lookup
2. a->ob_fval              <- unbox
3. b->ob_fval              <- unbox
4. double + double         <- 1 инструкция CPU  (полезная работа)
5. PyFloat_FromDouble()    <- malloc + GC
6. INCREF / DECREF         <- атомарные операции
```

С `cdef double a, b` шаги 1, 5, 6 исчезают полностью.

---
## 2  Hello World — первая `%%cython` ячейка

> **Требует C-компилятор.** Если его нет — ячейка упадёт с ошибкой, пропустите её.

In [ ]:
%%cython --quiet
# Это уже Cython-код — компилируется в C прямо сейчас

def hello(name: str) -> str:
    return f'Hello from Cython, {name}!'

In [ ]:
# Если %%cython --quiet сработал:
# print(hello('World'))
# print(type(hello))  # builtin_function_or_method — не Python-функция
print('Если видите эту строку — ячейка выше не запускалась')

---
## 3  `cdef` — статические типы

### Уровень 0 — чистый Python

```python
def py_sum_squares(n):
    s = 0.0          # s — PyObject* на куче
    for i in range(n):
        s += i * i   # PyNumber_Multiply + PyNumber_Add каждый раз
    return s
```

### Уровень 1 — Cython без типов (скомпилирован, но PyObject остаётся)

```cython
def cy_sum_squares_untyped(n):
    s = 0.0          # всё ещё PyObject*
    for i in range(n):
        s += i * i   # всё ещё PyNumber_*
    return s
```

### Уровень 2 — Cython typed (C-переменные на стеке)

```cython
def cy_sum_squares_typed(int n):
    cdef long long s = 0   # стек, 8 байт, не PyObject
    cdef int i
    for i in range(n):
        s += i * i         # одна инструкция CPU
    return s
```

### Что генерирует Cython

**Без типов** — из реального файла `src/cy_untyped/matrix_ops_cy.c`:
```c
PyObject *__pyx_v_s = NULL;           // heap!
__pyx_t_1 = PyNumber_Multiply(i, i);  // vtable dispatch
__pyx_v_s = PyNumber_Add(s, t1);      // malloc + GC
```

**С типами** — из реального файла `src/cy_typed/matrix_ops_typed.c`:
```c
double __pyx_v_s = 0.0;               // стек, 8 байт
__pyx_v_s += __pyx_v_i * __pyx_v_i;  // одна инструкция CPU
```

---
## 4  `def` / `cdef` / `cpdef` — три вида функций

```cython
# def — обычная Python-функция
def py_func(double x):
    return x * x

# cdef — только C-вызов, НЕ видна из Python
cdef double c_func(double x):
    return x * x

# cpdef — C-функция + автоматическая Python-обёртка
cpdef double cp_func(double x):
    return x * x

# Вызов c_func из Cython-кода (прямой C-вызов, нет overhead)
def call_c_func(double x):
    return c_func(x)
```

| | `def` | `cdef` | `cpdef` |
|---|---|---|---|
| Вызов из Python | да | **нет** | да |
| Вызов из Cython | да (медленно) | да (быстро) | да (быстро) |
| Overhead при вызове | Python API | нет | нет (из Cython) |
| Как callback | да | нет | да |

---
## 5  Typed Memoryviews — прямой доступ к буферу NumPy

`double[:] a` — структура с тремя полями: указатель, шаг, размер.
Доступ `a[i]` компилируется в `*(ptr + i * stride)` — одна инструкция.

```cython
# Без типов — каждый a[i] вызывает __getitem__, boxing
def dot_slow(a, b):
    s = 0.0
    for i in range(len(a)):
        s += a[i] * b[i]   # PyObject каждый раз
    return s

# Typed memoryview — прямой указатель
def dot_fast(double[:] a, double[:] b):
    cdef int n = a.shape[0]
    cdef int i
    cdef double s = 0.0
    for i in range(n):
        s += a[i] * b[i]   # *(ptr + i*stride)
    return s
```

**Синтаксис:**
```cython
double[:]       a   # 1D, любой шаг
double[::1]     a   # 1D, C-contiguous (stride=itemsize)
double[:, :]    A   # 2D, любые шаги
double[:, ::1]  A   # 2D, C-contiguous (строки подряд)
```

**Важно:** передавать `np.ascontiguousarray(arr, dtype=np.float64)` — иначе `BufferError`.

---
## 6  `nogil` и директивы компилятора

### nogil

```cython
def dot_nogil(double[::1] a, double[::1] b):
    cdef int n = a.shape[0]
    cdef int i
    cdef double s = 0.0
    with nogil:          # отпускаем GIL
        for i in range(n):
            s += a[i] * b[i]
    # GIL возвращается автоматически
    return s
```

Пока работает `with nogil:` — другие Python-потоки могут выполняться параллельно.

### Директивы компилятора

```cython
# cython: boundscheck=False   -- убирает проверку 0 <= i < n
# cython: wraparound=False    -- убирает поддержку a[-1]
# cython: cdivision=True      -- убирает проверку деления на 0
```

| Директива | Что убирает | Риск |
|-----------|-------------|------|
| `boundscheck=False` | Проверку `0 <= i < n` | Segfault при выходе за границы |
| `wraparound=False` | Поддержку `a[-1]` | Неверный результат |
| `cdivision=True` | Проверку `/0` | Undefined behavior |

---
## 7  Бенчмарки — 4 уровня оптимизации

Сравниваем **4 реализации** `sum_of_squares(n)` — суммы квадратов 0²+1²+…+(n-1)²:

| Уровень | Реализация | Ожидаемый speedup |
|---------|-----------|-------------------|
| 0 | Pure Python | 1x (baseline) |
| 1 | Cython untyped | ~1.3x (boxing остаётся) |
| 2 | Cython typed | ~50-200x (C-переменные) |
| 3 | NumPy (C+BLAS) | ~20x (векторизация) |

**Запустите ячейки по порядку.** Ячейки `%%cython` требуют C-компилятора.
Если компилятора нет — пропустите их, секция 7b работает без компилятора.

In [ ]:
import time
import numpy as np

def py_sum_squares(n):
    """Уровень 0: чистый Python — все переменные PyObject*"""
    s = 0.0
    for i in range(n):
        s += i * i
    return s

def bench(fn, *args, repeats=7):
    """Возвращает лучшее время из repeats запусков (мс)."""
    best = float('inf')
    for _ in range(repeats):
        t0 = time.perf_counter()
        fn(*args)
        best = min(best, time.perf_counter() - t0)
    return best * 1000

N = 500_000
t_py = bench(py_sum_squares, N)
print(f'Pure Python  sum_squares(N={N:,}): {t_py:.2f} ms  (1.0x baseline)')

### Уровень 1 — Cython untyped

Тот же код, скомпилированный Cython. Переменные `s` и `i` — всё ещё `PyObject*`.
Ожидаем ~1.3x — убирается только dispatch loop интерпретатора.

In [ ]:
%%cython --quiet
# Уровень 1: Cython untyped — переменные всё ещё PyObject*
# Cython компилирует это в C, но s и i остаются PyObject*

def cy_sum_squares_untyped(n):
    s = 0.0          # PyObject* на куче, 24 байта
    for i in range(n):
        s += i * i   # PyNumber_Multiply + PyNumber_Add каждый раз
    return s

In [ ]:
t_cy_u = bench(cy_sum_squares_untyped, N)
print(f'Cython untyped sum_squares(N={N:,}): {t_cy_u:.2f} ms  ({t_py/t_cy_u:.1f}x)')
print('  → boxing PyObject* остаётся → небольшой прирост')

### Уровень 2 — Cython typed

`cdef long long s` и `cdef int i` — C-переменные на стеке, не PyObject.
Ожидаем ~50-200x — boxing исчезает полностью.

In [ ]:
%%cython --quiet
# Уровень 2: Cython typed — C-переменные на стеке
# cython: boundscheck=False
# cython: wraparound=False

def cy_sum_squares_typed(int n):
    cdef long long s = 0   # стек, 8 байт, не PyObject
    cdef int i             # стек, 4 байта
    for i in range(n):     # компилируется в C for-loop
        s += i * i         # одна инструкция IMUL
    return s               # boxing только при return

In [ ]:
t_cy_t = bench(cy_sum_squares_typed, N)
print(f'Cython typed   sum_squares(N={N:,}): {t_cy_t:.3f} ms  ({t_py/t_cy_t:.0f}x)')
print('  → C-переменные на стеке → boxing исчезает → максимальный speedup')

### Уровень 3 — NumPy (C + векторизация)

NumPy использует BLAS и SIMD-инструкции. Для скалярных циклов Cython typed быстрее,
для матричных операций NumPy выигрывает за счёт BLAS.

In [ ]:
def np_sum_squares(n):
    """NumPy: arange + dot — векторизованный C"""
    a = np.arange(n, dtype=np.float64)
    return float(np.dot(a, a))

t_np = bench(np_sum_squares, N)
print(f'NumPy          sum_squares(N={N:,}): {t_np:.3f} ms  ({t_py/t_np:.0f}x)')
print()
print('─' * 60)
print(f'Итог sum_squares(N={N:,}):')
print(f'  Pure Python    : {t_py:8.2f} ms  1.0x')
try:
    print(f'  Cython untyped : {t_cy_u:8.2f} ms  {t_py/t_cy_u:.1f}x')
    print(f'  Cython typed   : {t_cy_t:8.3f} ms  {t_py/t_cy_t:.0f}x')
except NameError:
    print('  Cython untyped : (не скомпилирован)')
    print('  Cython typed   : (не скомпилирован)')
print(f'  NumPy          : {t_np:8.3f} ms  {t_py/t_np:.0f}x')

### График — сравнение всех 4 уровней

In [ ]:
import matplotlib.pyplot as plt

labels = ['Pure Python', 'Cython\nuntyped', 'Cython\ntyped', 'NumPy']
try:
    times = [t_py, t_cy_u, t_cy_t, t_np]
    colors = ['#e74c3c', '#e67e22', '#2ecc71', '#3498db']
except NameError:
    # Если Cython не скомпилирован — показываем только Python и NumPy
    labels = ['Pure Python', 'NumPy']
    times = [t_py, t_np]
    colors = ['#e74c3c', '#3498db']

speedups = [t_py / t for t in times]

fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(12, 4))

bars = ax1.bar(labels, times, color=colors, edgecolor='black', alpha=0.85)
ax1.set_ylabel('Время (мс, log scale)')
ax1.set_title(f'sum_squares(N={N:,}) — время выполнения')
ax1.set_yscale('log')
ax1.grid(axis='y', alpha=0.3)
for bar, t in zip(bars, times):
    ax1.text(bar.get_x() + bar.get_width()/2, bar.get_height() * 1.1,
             f'{t:.2f}ms', ha='center', va='bottom', fontsize=9)

bars2 = ax2.bar(labels, speedups, color=colors, edgecolor='black', alpha=0.85)
ax2.set_ylabel('Ускорение (x раз)')
ax2.set_title('Speedup относительно Pure Python')
ax2.axhline(1, color='red', linestyle='--', alpha=0.5, label='baseline')
ax2.grid(axis='y', alpha=0.3)
for bar, s in zip(bars2, speedups):
    ax2.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 0.5,
             f'{s:.1f}x', ha='center', va='bottom', fontsize=9, fontweight='bold')

plt.tight_layout()
plt.show()

---
## 7b  Память: boxing vs C-типы

In [ ]:
import tracemalloc
import numpy as np

N = 128

# Python lists — каждый float = PyFloatObject = 24 байта
tracemalloc.start()
A_ls = [[float(i*N+j) for j in range(N)] for i in range(N)]
_, peak_list = tracemalloc.get_traced_memory()
tracemalloc.stop()

# NumPy array — каждый float = double = 8 байт
tracemalloc.start()
A_np = np.arange(N*N, dtype=np.float64).reshape(N, N)
_, peak_np = tracemalloc.get_traced_memory()
tracemalloc.stop()

print(f'Матрица {N}x{N} = {N*N:,} элементов')
print(f'  Python list of lists : {peak_list/1024:7.1f} KB  ({peak_list/(N*N):.1f} байт/элемент)')
print(f'  NumPy float64        : {peak_np/1024:7.1f} KB  ({peak_np/(N*N):.1f} байт/элемент)')
print(f'  Теоретически: PyFloatObject=24B, double=8B')
print(f'  Экономия памяти: {peak_list/peak_np:.1f}x')

---
## 8  Ограничения Cython

| Ограничение | Подробности |
|---|---|
| **Обязательный build step** | `.pyx` нужно компилировать; нельзя просто запустить |
| **Нужен C-компилятор** | MSVC на Windows, gcc/clang на Linux/macOS |
| **Отладка сложна** | Трейсбеки указывают на сгенерированный C, не на `.pyx` |
| **GIL нужен для Python-объектов** | `nogil` работает только если все переменные — C-типы |
| **Нет JIT** | В отличие от PyPy/Numba, не оптимизирует во время выполнения |
| **Typed memoryviews требуют contiguous** | Нельзя передать произвольный срез |
| **Версионная совместимость** | Расширение для Python 3.10 может не загрузиться на 3.12 |

### Когда использовать Cython

| Сценарий | Лучший инструмент |
|---|---|
| Тесные числовые циклы | **Cython typed** |
| Обёртка C/C++ библиотеки | **Cython** или **pybind11** |
| Матричная математика | **NumPy** (BLAS, SIMD) |
| JIT без шага сборки | **Numba** `@njit` |
| GPU | **CuPy / JAX** |
| Максимальная скорость интерпретатора | **PyPy** |

---
## Итог

```
Pure Python  -->  Cython untyped  -->  Cython typed  -->  C
   1x               ~1.3x              ~100-200x       ~100-250x
```

- **Cython untyped** — убирает dispatch loop, boxing остаётся → ~1.3x
- **Cython typed** — убирает всё: boxing, GC, GIL → ~100-200x
- **C** — разница с Cython typed < 5%; Cython генерирует идентичный C

**Правило:** профилируй → найди горячий цикл → добавь `cdef` + memoryview → измерь